<a href="https://www.kaggle.com/code/chiro999/transformercustom?scriptVersionId=309315492" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# import OS and check if dataset is mounted
import os

print("Datasets mounted at /kaggle/input:")
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets")[:50])
print(os.listdir("/kaggle/input/datasets/organizations")[:100])

Datasets mounted at /kaggle/input:
['datasets']
['organizations']
['nih-chest-xrays']


In [2]:
# Storing root path of ChestXray14 dataset
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays"
print(os.listdir(DATA_ROOT)[:200])

['data']


In [3]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

first_png = None
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith(".png"):
            first_png = os.path.join(root, f)
            break
    if first_png:
        break

print("First PNG found:", first_png)


First PNG found: /kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images/00006199_010.png


In [4]:
# Checking whether a CUDA capable GPU is available
!nvidia-smi

Mon Apr  6 05:22:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# importing timm for Vision Transformer 
!pip -q install timm

# Importing PIL to open images and tqdm for progress bars as well as random for reproducibilty
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# PyTorch and its neural network tools.
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# timm for loading the ViT model
import timm
from torchvision import transforms

# auroc metric from sklearn
from sklearn.metrics import roc_auc_score

In [6]:
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [7]:
# Pointing to the metadata csv that contains image names, patient IDs, and label
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

# The 14 chest disease classes
LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]

# Creating a dictionary for these labels
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

# use a same seed to reproduce results
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# Select GPU if available otherwise GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [8]:
# collect all subdirectories inside DATA_ROOT that match 'images_' and contain an 'images' folder
img_dirs = [
    os.path.join(DATA_ROOT, d, "images")
    for d in os.listdir(DATA_ROOT)
    if d.startswith("images_") and os.path.isdir(os.path.join(DATA_ROOT, d, "images"))
]

# print number of image directories found and preview a few paths
print("Image folders found:", len(img_dirs))
print("Example dirs:", img_dirs[:3])

# create a dictionary mapping image filenames to their full file paths
name2path = {}
for d in img_dirs:
    for f in os.listdir(d):
        if f.endswith(".png"):
            name2path[f] = os.path.join(d, f)
        
# print total number of indexed images and check if a sample file exists
print("Indexed images:", len(name2path))
print("Has 00000001_000.png?", "00000001_000.png" in name2path)

Image folders found: 12
Example dirs: ['/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_012/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_009/images']
Indexed images: 112120
Has 00000001_000.png? True


In [9]:
# load dataset from CSV file into a DataFrame
df = pd.read_csv(CSV_PATH)

# get unique patient IDs and shuffle them for random splitting
patients = df["Patient ID"].unique()
np.random.shuffle(patients)

# split sizes (80% train, 10% validation, 10% test)
n = len(patients)
train_ids = set(patients[: int(0.8*n)])
val_ids   = set(patients[int(0.8*n): int(0.9*n)])
test_ids  = set(patients[int(0.9*n):])

# split the DataFrame based on patient IDs to avoid data leakage
train_df = df[df["Patient ID"].isin(train_ids)].copy()
val_df   = df[df["Patient ID"].isin(val_ids)].copy()
test_df  = df[df["Patient ID"].isin(test_ids)].copy()

# print number of samples in each split
print(len(train_df), len(val_df), len(test_df))

89703 11221 11196


In [10]:
class ChestXray14(Dataset):
    # store dataframe and reset index for consistent access
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    # return total number of samples
    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        # get row corresponding to index i
        row = self.df.iloc[i]

        # extract image filename and retrieve full path
        fname = row["Image Index"]
        img_path = name2path[fname] 

        # load image in grayscale and convert to 3-channel RGB
        img = Image.open(img_path).convert("L")
        img = Image.merge("RGB", (img, img, img))

        # initialize multi-label target vector
        y = np.zeros(NUM_LABELS, dtype=np.float32)
         # set labels to 1 for each disease present
        for f in str(row["Finding Labels"]).split("|"):
            if f in label2idx:
                y[label2idx[f]] = 1.0
                
        # apply data transformations if provided
        if self.transform:
            img = self.transform(img)
            
        # return image tensor and corresponding label tensor
        return img, torch.tensor(y)

IMG_SIZE = 224
# training data transformations (data augmentation + normalization)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

# validation data transformations (no augmentation, only resizing and normalization)
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

In [11]:
# define batch size and number of parallel workers for data loading
BATCH_SIZE = 32
NUM_WORKERS = 2

# create DataLoader for training set with shuffling for randomness
train_loader = DataLoader(ChestXray14(train_df, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

# create DataLoader for validation set (no shuffling for consistent evaluation)
val_loader = DataLoader(ChestXray14(val_df, val_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# create DataLoader for test set (no shuffling, same as validation) 
test_loader = DataLoader(ChestXray14(test_df, val_tf),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

In [12]:
def compute_pos_weight(df_):
    # initialize count of positive samples for each label
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    # count occurrences of each label in the dataset
    for labels in df_["Finding Labels"].astype(str).values:
        for f in labels.split("|"):
            if f in label2idx:
                counts[label2idx[f]] += 1
    # total number of samples
    N = len(df_)
    # ensure no division by zero by clipping minimum positive count to 1
    pos = np.clip(counts, 1.0, None)
    # compute number of negative samples per label
    neg = N - counts
    # return class weights (neg/pos) to address class imbalance
    return torch.tensor(neg / pos, dtype=torch.float32)

# compute class weights from training data and move to device (CPU/GPU)
pos_weight = compute_pos_weight(train_df).to(device)

# print computed class weights
print("pos_weight:", pos_weight)

pos_weight: tensor([  8.7609,  39.6448,   7.5310,   4.5925,  18.7889,  16.7630,  76.3969,
         19.5694,  23.1332,  48.3959,  42.3767,  63.4418,  32.6344, 466.2031],
       device='cuda:0')


In [13]:
import torch
import torch.nn as nn

# module to split image into patches and project to embedding space
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        # compute number of patches
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        # convolution to extract and embed patches
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # (B, C, H, W) -> (B, N, D)
        # convert image to patch embeddings
        x = self.proj(x)                    
        # reshape to sequence of patches
        x = x.flatten(2).transpose(1, 2)    # (B, N, D)
        return x

# feed-forward network used inside transformer block
class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x

# single transformer encoder block (attention + MLP)
class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, attn_drop=0.0, drop=0.0):
        super().__init__()
        # layer normalization before attention
        self.norm1 = nn.LayerNorm(dim)
        # multi-head self-attention layer
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=attn_drop, batch_first=True)
        self.drop1 = nn.Dropout(drop)
        # layer normalization before MLP
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio=mlp_ratio, drop=drop)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x):
        # self-attention with residual connection
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, need_weights=False)
        x = x + self.drop1(attn_out)

        # MLP with residual connection
        x = x + self.drop2(self.mlp(self.norm2(x)))
        return x

# custom Vision Transformer model
class CustomViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        num_classes=14,
        embed_dim=256,
        depth=6,
        num_heads=8,
        mlp_ratio=4.0,
        drop=0.1,
        attn_drop=0.1,
    ):
        super().__init__()
        # patch embedding module
        self.patch = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        n = self.patch.num_patches

        # learnable classification token and positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop)

        # stack of transformer encoder blocks
        self.blocks = nn.ModuleList([
            EncoderBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, attn_drop=attn_drop, drop=drop)
            for _ in range(depth)
        ])
        # final normalization and classification head
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

         # initialize learnable parameters
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
         # convert image to patch embeddings
        x = self.patch(x)  # (B, N, D)
        B = x.size(0)
        # expand classification token for batch
        cls = self.cls_token.expand(B, -1, -1)
        # prepend class token to patch sequence# 
        x = torch.cat([cls, x], dim=1)         
        # add positional embeddings and apply dropout
        x = x + self.pos_embed
        x = self.pos_drop(x)
        # pass through transformer encoder blocks
        for blk in self.blocks:
            x = blk(x)
        # apply final normalization
        x = self.norm(x)
        # extract classification token representation
        cls_out = x[:, 0]
        # compute output logits for classification
        logits = self.head(cls_out)  
        return logits


In [14]:
# ChestX-ray14 labels
NUM_LABELS = len(LABELS)

# initialize custom Vision Transformer model 
model = CustomViT(
    img_size=224,
    patch_size=16,
    in_chans=3,
    num_classes=NUM_LABELS,
    embed_dim=384,   
    depth=6,
    num_heads=8,
    mlp_ratio=4.0,
    drop=0.1,
    attn_drop=0.1,
).to(device)

print("Using CustomViT from scratch ")


Using CustomViT from scratch 


In [15]:
@torch.no_grad() # disable gradient computation for evaluation
def eval_auc_per_class(model, loader, device, label_names):
    """Returns (mean_auc, per_class_aucs_list) aligned with label_names."""
    # set model to evaluation mode
    model.eval()
    all_logits, all_y = [], []

    # iterate over dataset batches
    for x, y in tqdm(loader, leave=False):
        # move inputs to device
        x = x.to(device, non_blocking=True)
        logits = model(x).float().cpu()
        all_logits.append(logits)
        all_y.append(y.cpu())
    # concatenate all predictions and ground truth labels
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy()
    # apply sigmoid to convert logits to probabilities
    y_prob = 1 / (1 + np.exp(-logits))  # sigmoid

    # compute AUROC for each class
    per_class = []
    for i in range(len(label_names)):
        # skip classes with only one label present (undefined AUC)
        if len(np.unique(y_true[:, i])) < 2:
            per_class.append(np.nan)
        else:
            per_class.append(roc_auc_score(y_true[:, i], y_prob[:, i]))

    # compute mean AUROC across all classes (ignoring NaN values)
    mean_auc = float(np.nanmean(per_class))
    return mean_auc, per_class


In [16]:
def compute_pos_weight(df_):
    # initialize count of positive samples for each label
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    # count occurrences of each label in the dataset
    for labels in df_["Finding Labels"].astype(str).values:
        labs = labels.split("|")
        for f in labs:
            if f in label2idx:
                counts[label2idx[f]] += 1
    # total number of samples        
    N = len(df_)
    # clip positive counts to avoid division by zero
    pos = np.clip(counts, 1.0, None)
     # compute number of negative samples per label
    neg = N - counts
    # return class weights (neg/pos) for handling class imbalance
    return torch.tensor(neg/pos, dtype=torch.float32)

# compute class weights from training data and move to device
pos_weight = compute_pos_weight(train_df).to(device)

# define multi-label loss function with class imbalance weighting
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [17]:
# define optimizer (AdamW with learning rate and weight decay for regularization)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

# define learning rate scheduler using cosine annealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

# initialize gradient scaler for mixed precision training
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

/tmp/ipykernel_24/1049078374.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))


In [18]:
def train_one_epoch():
    # set model to training mode
    model.train()
    total_loss = 0.0
    # iterate over training batches
    for x, y in tqdm(train_loader, leave=False):
        # move inputs and labels to device (GPU)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # reset gradients
        optimizer.zero_grad(set_to_none=True)
        # forward pass with mixed precision
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        # backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # accumulate total loss (scaled by batch size)
        total_loss += loss.item() * x.size(0)

     # return average loss over entire training dataset
    return total_loss / len(train_loader.dataset)


In [19]:
# define number of training epochs
EPOCHS = 5
# initialize best validation score
best_val = -1

# path to save the best model checkpoint
best_path = "/kaggle/working/best_custom_vit_chestxray14.pt"

# training loop over epochs
for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch()
    val_mean_auc, _ = eval_auc_per_class(model, val_loader, device, LABELS)

     # print training progress for current epoch
    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} | val mean AUROC {val_mean_auc:.4f}")

    # save model checkpoint if validation performance improves
    if val_mean_auc > best_val:
        best_val = val_mean_auc
        torch.save(model.state_dict(), best_path)
        print("saved best:", best_path)

# print best validation AUROC achieved during training
print("Best val mean AUROC:", best_val)


  0%|          | 0/2804 [00:00<?, ?it/s]/tmp/ipykernel_24/3916925165.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 1/5 | train loss 1.3336 | val mean AUROC 0.5625
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


  0%|          | 0/2804 [00:00<?, ?it/s]/tmp/ipykernel_24/3916925165.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 2/5 | train loss 1.2971 | val mean AUROC 0.5894
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


  0%|          | 0/2804 [00:00<?, ?it/s]/tmp/ipykernel_24/3916925165.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 3/5 | train loss 1.2917 | val mean AUROC 0.5928
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


  0%|          | 0/2804 [00:00<?, ?it/s]/tmp/ipykernel_24/3916925165.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 4/5 | train loss 1.2848 | val mean AUROC 0.5881


  0%|          | 0/2804 [00:00<?, ?it/s]/tmp/ipykernel_24/3916925165.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 5/5 | train loss 1.2791 | val mean AUROC 0.6010
saved best: /kaggle/working/best_custom_vit_chestxray14.pt
Best val mean AUROC: 0.6010056274958152


In [20]:
# load the best saved model weights
model.load_state_dict(torch.load(best_path, map_location=device))
# evaluate model on the test set
test_mean_auc, test_aucs = eval_auc_per_class(model, test_loader, device, LABELS)

# print overall test performance (mean AUROC)
print("Test mean AUROC:", test_mean_auc)

# print AUROC score for each class
for name, auc in zip(LABELS, test_aucs):
    print(f"{name:18s}: {auc:.4f}")


Test mean AUROC: 0.6075989410248192
Atelectasis       : 0.5957
Cardiomegaly      : 0.5987
Effusion          : 0.6280
Infiltration      : 0.6168
Mass              : 0.5234
Nodule            : 0.5711
Pneumonia         : 0.6211
Pneumothorax      : 0.5992
Consolidation     : 0.6976
Edema             : 0.7554
Emphysema         : 0.4949
Fibrosis          : 0.6141
Pleural_Thickening: 0.5124
Hernia            : 0.6779
